# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Pranajit04/Machine-Learning-Flyrank/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
%pip -q install pandas
import pandas as pd

df = pd.read_csv('https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv')
print(df.shape)
df.head()


(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 1. My lane as an ML task

**Task type:** Binary classification.

I am predicting whether a content page is "declining" (needs review) or "not declining" —
a yes/no label, not a number, not a group. This is classification because the output is
a discrete category a reviewer can act on directly.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

## 2. Target or proxy

**Target:** `is_declining` — derived from the `trend_direction` column, where
`is_declining = (trend_direction == "down")`.

This is a proxy, not ground truth — no human has confirmed "this page needs
review." It's built from `impressions_last_30d` vs `impressions_prev_30d`
momentum, already computed in this dataset as `trend_direction`/`trend_pct`.
I use the label as-is rather than `trend_pct` or the 30-day columns directly
as features, since those columns define the label and using them as inputs
too would be leakage.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Confirm the label source and check class balance
print(df['trend_direction'].value_counts())
print()
df['is_declining'] = (df['trend_direction'] == 'down').astype(int)
print("is_declining balance:")
print(df['is_declining'].value_counts(normalize=True))


trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

is_declining balance:
is_declining
1    0.542067
0    0.457933
Name: proportion, dtype: float64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

## 3. Success metric

**Metric:** Precision on the "declining" class, compared against a simple
hand-rule baseline, evaluated with a client-level train/test split.

Precision matters more than raw accuracy here because a reviewer's time is
limited — a false positive (flagging a healthy page as declining) wastes
review time, so I want confidence when the model does flag something.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Confirm client_id exists so a client-level split is actually possible
print("Unique clients:", df['client_id'].nunique())
print("Rows per client (sample):")
print(df['client_id'].value_counts().head())


Unique clients: 32
Rows per client (sample):
client_id
client_19581e27de    7008
client_6208ef0f77    3681
client_4e07408562    2294
client_3fdba35f04    2267
client_f369cb89fc    1796
Name: count, dtype: int64


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# One row = one content item (identified by content_id, tied to a client_id)
df[['content_id', 'client_id', 'impressions_last_30d', 'clicks_last_30d',
    'avg_position', 'trend_direction']].head(10)

,content_id,client_id,impressions_last_30d,clicks_last_30d,avg_position,trend_direction
0,content_304f48230142,client_f369cb89fc,578,2,10.6,down
1,content_a1fb4e703a9e,client_4e07408562,2501,2,20.3,down
2,content_9aa793d4d895,client_7f2253d7e2,2382,1,36.5,down
3,content_331d6c4de07b,client_19581e27de,3626,22,6.2,stable
4,content_d99b7a2d90ca,client_3fdba35f04,4211,10,44.0,down
5,content_d4084a4bc775,client_f369cb89fc,617,0,8.5,down
6,content_9a34b442b552,client_8722616204,1,0,7.0,down
7,content_a63219c6e95a,client_19581e27de,636,1,21.2,stable
8,content_5e6c160719bc,client_6208ef0f77,5696,9,46.0,down
9,content_c27558df2b0c,client_19581e27de,252,0,4.9,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

## 5. Why ML beats a fixed rule here

A fixed rule (e.g. "flag if impressions dropped 20%") only looks at one signal
at a time. Real decline often shows up as a *combination* — a small impression
drop plus rising position volatility plus a shrinking query footprint — that a
single threshold can't capture. A model can learn which combinations of
signals matter most (and how much), which is exactly what feature importance
showed in my capstone work: position volatility and query concentration
mattered nearly as much as the raw impression trend.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Quick check: do individual signals correlate weakly with each other?
# If they did perfectly, a simple rule would be enough — weak correlation supports needing ML
print(df[['search_volume', 'competition', 'cpc', 'word_count']].corr())

               search_volume  competition       cpc  word_count
search_volume       1.000000     0.049887  0.042085   -0.019458
competition         0.049887     1.000000  0.302415   -0.201019
cpc                 0.042085     0.302415  1.000000   -0.090349
word_count         -0.019458    -0.201019 -0.090349    1.000000


## 6. Self-check

- [x] Named the task type: binary classification
- [x] Named the target/proxy: is_declining, built from impressions + clicks trend
- [x] Named the success metric: precision vs. hand-rule baseline, client-split
- [x] Showed the unit of analysis as a real dataframe (one row = one content page)
- [x] Explained why ML beats a fixed rule: signal combinations, not single thresholds

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.